# AIGC 5005 · Module 3 · Part 1
## Classes, instances, attributes, and methods

Companion notebook to Part 1 of the Module 3 learning content. Every example from that part is here, ready to run, with comments explaining what to notice.

**Estimated time:** 30 minutes

**How to use it:** read each explanation, run the cell beneath it, and compare the output with what you expected *before* you ran it. The prediction is where the learning happens. Change values and run cells again; you cannot break anything.

**Kernel:** select the kernel from your `module03` conda environment (top right in VS Code or Jupyter). To make one: `conda create -n module03 python=3.11 ipykernel`, then `conda activate module03`. If VS Code offers to install `ipykernel`, accept.

## 1. Defining a class

A **class** is a blueprint. An **instance** is one object built from it. `__init__` runs automatically when you create an instance and sets up that instance's **attributes**. Every method receives the instance as its first parameter, conventionally called `self`.

In [ ]:
class Book:
    """A single book in the catalogue."""

    def __init__(self, title, author, copies=1):
        self.title = title          # instance attributes: each Book gets its own copy
        self.author = author
        self.copies = copies

    def checkout(self):
        """Lend one copy, or raise ValueError if none are left."""
        if self.copies <= 0:
            raise ValueError(f"No copies of '{self.title}' available.")
        self.copies -= 1

    def __str__(self):
        # Python calls this automatically for print(book) and str(book)
        return f"{self.title} by {self.author} ({self.copies} available)"

### Create two instances and use them

**Predict first:** what will the last line print?

In [ ]:
b1 = Book("Fluent Python", "Luciano Ramalho", copies=2)
b2 = Book("Clean Code", "Robert Martin", copies=1)

print(b1)
print(b2)

b1.checkout()
print(b1)               # only b1 changed

b2.checkout()
try:
    b2.checkout()       # no copies left, so this raises ValueError
except ValueError as error:   # try/except is only here to keep the notebook running; it is formally taught in Module 4
    print(f"Caught: {error}")

print(b1.copies, b2.copies)   # each instance keeps its own attributes: 1 0

### `self` is just the instance the method was called on

`b1.checkout()` is shorthand for `Book.checkout(b1)`. Run both and compare.

In [ ]:
c = Book("Refactoring", "Martin Fowler", copies=3)
c.checkout()                 # the everyday way
Book.checkout(c)             # the same call written out: self is c
print(c.copies)              # 1

### Watch out: forgetting `self`

If a method is defined without `self`, Python still passes the instance in, so you get a confusing `TypeError` about an *extra* argument. The `try/except` below just lets the notebook keep running.

In [ ]:
class Broken:
    def greet():                 # BUG: no self parameter
        return "hello"


try:
    Broken().greet()
except TypeError as error:
    print("TypeError:", error)   # takes 0 positional arguments but 1 was given

### Try it

Write a `Student` class with attributes `name` and a class attribute `school = "Humber Polytechnic"`. Add a method `enrol(course)` that appends to a per-instance list. Create two students, enrol each in different courses, and confirm their course lists are independent. (Sections 2 and 3 below show the pieces you need.)

## 2. Class attributes versus instance attributes

An attribute set in the class body is a **class attribute**, shared by every instance. An attribute set with `self.` is an **instance attribute**, owned by one object.

In [ ]:
class Book:
    library_name = "Humber Learning Commons"   # class attribute: shared by all
    total_books = 0                             # a shared counter

    def __init__(self, title, copies=1):
        self.title = title
        self.copies = copies
        Book.total_books += 1                   # update the class attribute through the class name


b1 = Book("Fluent Python")
b2 = Book("Clean Code")

print(Book.total_books)                          # 2
print(b1.library_name, "|", b2.library_name)     # both read the shared value

### Shadowing: assigning through an instance does not change the class

Python looks on the instance first, then on the class. Assigning to `b1.library_name` creates a *new* instance attribute that hides (shadows) the class one for `b1` only.

**Predict first:** what will each of the three values be?

In [ ]:
b1.library_name = "North Campus Library"
print(b1.library_name)       # North Campus Library  (instance attribute, found first)
print(b2.library_name)       # Humber Learning Commons (b2 still uses the class value)
print(Book.library_name)     # Humber Learning Commons (the class was never changed)

print("library_name" in vars(b1))   # True: it now lives on the instance
print("library_name" in vars(b2))   # False: b2 has no such instance attribute

### Watch out: a mutable class attribute is shared by everyone

A list created in the class body is **one** list, shared by every instance. Create per-instance lists inside `__init__`.

In [ ]:
class BadStudent:
    courses = []                    # ONE list, shared by every BadStudent

    def enrol(self, course):
        self.courses.append(course)


class GoodStudent:
    def __init__(self):
        self.courses = []           # a NEW list for each student

    def enrol(self, course):
        self.courses.append(course)


a, b = BadStudent(), BadStudent()
a.enrol("AIGC 5005")
print("BadStudent b.courses :", b.courses)    # b never enrolled, but sees a's course

c, d = GoodStudent(), GoodStudent()
c.enrol("AIGC 5005")
print("GoodStudent d.courses:", d.courses)    # independent, as intended

## 3. Inspecting objects

`vars(obj)` shows an instance's own attributes, and `type(obj)` shows its class. Both are handy when something is not behaving the way you expect.

In [ ]:
book = Book("Clean Code", copies=4)
print(vars(book))            # {'title': 'Clean Code', 'copies': 4}
print(type(book))            # <class '__main__.Book'>
print(type(book).__name__)   # Book